In [ ]:
import pandas as pd
import numpy as np

from skeLCS.eLCS import eLCS

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

data_raw = pd.read_csv("../data/weatherAUS.csv")

print("Original shape:", data_raw.shape)
print(data_raw.head())


In [ ]:
# remove rows where the target is missing
data_raw = data_raw.dropna(
    subset=["RainTomorrow"]
).copy()

# convert target to numeric
data_raw["RainTomorrow"] = data_raw["RainTomorrow"].map({
    "No": 0,
    "Yes": 1
})

# Date is removed for the raw baseline rather than feature engineered
data_raw = data_raw.drop(columns=["Date"])

print("Shape after removing missing target:", data_raw.shape)

print("\nTarget distribution:")
print(data_raw["RainTomorrow"].value_counts())

print("\nMissing feature values:")
print(data_raw.isna().sum().sort_values(ascending=False))


In [ ]:
classlabel = "RainTomorrow"

x_raw = data_raw.drop(columns=[classlabel])
y_raw = data_raw[classlabel]

x_train, x_test, y_train, y_test = train_test_split(
    x_raw,
    y_raw,
    test_size=0.20,
    random_state=42,
    stratify=y_raw
)

print("Training:", x_train.shape)
print("Testing:", x_test.shape)

print("\nTraining target:")
print(y_train.value_counts())

print("\nTesting target:")
print(y_test.value_counts())


In [ ]:
# categorical values must be numeric for eLCS
categorical_cols = x_train.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

print("Categorical columns:")
print(categorical_cols)

encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

x_train = x_train.copy()
x_test = x_test.copy()

x_train[categorical_cols] = encoder.fit_transform(
    x_train[categorical_cols]
)

x_test[categorical_cols] = encoder.transform(
    x_test[categorical_cols]
)


In [ ]:
# tell eLCS which encoded columns are discrete categories
discrete_attributes = np.array([
    x_train.columns.get_loc(col)
    for col in categorical_cols
])

print("Discrete attribute indices:")
print(discrete_attributes)

# professor's eLCS requires numeric numpy arrays
x_train_np = x_train.to_numpy(dtype=float)
x_test_np = x_test.to_numpy(dtype=float)

y_train_np = y_train.to_numpy(dtype=float)
y_test_np = y_test.to_numpy(dtype=float)

print("\nTraining shape:", x_train_np.shape)
print("Testing shape:", x_test_np.shape)
print("Features:", x_train_np.shape[1])

print("\nMissing training feature values:",
      np.isnan(x_train_np).sum())

print("Missing testing feature values:",
      np.isnan(x_test_np).sum())


In [9]:
# small raw eLCS test run before the full experiment
x_sample, _, y_sample, _ = train_test_split(
    x_train_np,
    y_train_np,
    train_size=25000,
    random_state=42,
    stratify=y_train_np
)

raw_test_model = eLCS(
    learning_iterations=5000,
    N=1000,
    discrete_attribute_limit="d",
    specified_attributes=discrete_attributes,
    random_state=42
)

raw_test_model.fit(
    x_sample,
    y_sample
)

print("Raw eLCS test training complete")


Raw eLCS test training complete


In [ ]:
raw_test_predictions = raw_test_model.predict(
    x_test_np
)

print(
    "Accuracy:",
    accuracy_score(
        y_test_np,
        raw_test_predictions
    )
)

print(
    "Balanced Accuracy:",
    balanced_accuracy_score(
        y_test_np,
        raw_test_predictions
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_test_np,
        raw_test_predictions
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test_np,
        raw_test_predictions
    )
)


In [ ]:
# full raw eLCS baseline
# learning iterations match the number of training instances
raw_elcs = eLCS(
    learning_iterations=len(x_train_np),
    N=1000,
    discrete_attribute_limit="d",
    specified_attributes=discrete_attributes,
    random_state=42
)

raw_elcs.fit(
    x_train_np,
    y_train_np
)

print("Raw eLCS training complete")


In [ ]:
raw_predictions = raw_elcs.predict(
    x_test_np
)

print(
    "Accuracy:",
    accuracy_score(
        y_test_np,
        raw_predictions
    )
)

print(
    "Balanced Accuracy:",
    balanced_accuracy_score(
        y_test_np,
        raw_predictions
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_test_np,
        raw_predictions
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test_np,
        raw_predictions
    )
)


In [ ]:
import os

os.makedirs("../results", exist_ok=True)

raw_results = pd.DataFrame({
    "actual": y_test_np,
    "prediction": raw_predictions
})

raw_results.to_csv(
    "../results/raw_elcs_predictions.csv",
    index=False
)

raw_elcs.export_iteration_tracking_data(
    "../results/raw_elcs_iterations.csv"
)

raw_elcs.export_final_rule_population(
    headerNames=x_train.columns.to_numpy(),
    className="RainTomorrow",
    filename="../results/raw_elcs_rules.csv"
)

print("Raw eLCS results saved")
